<a href="https://colab.research.google.com/github/gez2code/dermamnist-hybrid-study/blob/main/Experiment_Setup_Modular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# --- INSTALL & IMPORT ---
!pip install medmnist wandb -q
import os, random, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from medmnist import DermaMNIST
import wandb
from wandb.integration.keras import WandbMetricsLogger
import matplotlib.pyplot as plt
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# --- REPRODUCIBILITY ---
SEED = 42
def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)
set_seeds()

# --- DATA LOADER FUNCTION ---
def load_and_prep_data():
    print("Loading DermaMNIST...")
    train_data = DermaMNIST(split='train', download=True, size=28)
    val_data = DermaMNIST(split='val', download=True, size=28)
    test_data = DermaMNIST(split='test', download=True, size=28)

    # Extract & Normalize
    x_train, y_train = train_data.imgs.astype('float32')/255.0, train_data.labels
    x_val, y_val = val_data.imgs.astype('float32')/255.0, val_data.labels
    x_test, y_test = test_data.imgs.astype('float32')/255.0, test_data.labels

    # Binary Mapping (Illness vs Benign)
    to_binary = lambda y: np.isin(y, [0, 1, 6]).astype(int)
    y_train_bin, y_val_bin, y_test_bin = to_binary(y_train), to_binary(y_val), to_binary(y_test)

    # One-Hot Encode
    y_train_enc = tf.keras.utils.to_categorical(y_train_bin, 2)
    y_val_enc = tf.keras.utils.to_categorical(y_val_bin, 2)
    y_test_enc = tf.keras.utils.to_categorical(y_test_bin, 2)

    # Calculate Class Weights
    cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train_bin), y=y_train_bin.flatten())
    weights = {0: cw[0], 1: cw[1]}

    return (x_train, y_train_enc), (x_val, y_val_enc), (x_test, y_test_enc), weights

# Load Data Once
(x_train, y_train), (x_val, y_val), (x_test, y_test), class_weights = load_and_prep_data()

# --- TRAINER FUNCTION (With Interactive Test Viz) ---
def train_experiment(model_builder, exp_name, config):
    # 1. Init W&B
    if wandb.run is not None: wandb.finish()
    wandb.init(project="DermaMNIST_Project", name=exp_name, config=config)

    # 2. Build & Compile
    model = model_builder()
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

    # 3. Train
    datagen = ImageDataGenerator(rotation_range=10, width_shift_range=0.2, height_shift_range=0.2, shear_range=0.2, horizontal_flip=True, fill_mode='nearest')
    history = model.fit(
        datagen.flow(x_train, y_train, batch_size=config['batch_size'], seed=SEED),
        epochs=config['epochs'],
        validation_data=(x_val, y_val),
        class_weight=class_weights,
        callbacks=[WandbMetricsLogger()],
        verbose=1
    )

    # 4. EVALUATE & VISUALIZE
    print("Generating Test Predictions...")

    # Get raw probabilities (needed for ROC curve)
    y_pred_probs = model.predict(x_test)
    # Get hard class predictions (0 or 1)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    # Get true labels (0 or 1)
    y_true_classes = np.argmax(y_test, axis=1)

    # A. Log Scalar Metrics (So you can sort runs in the W&B Table)
    from sklearn.metrics import accuracy_score, roc_auc_score
    test_acc = accuracy_score(y_true_classes, y_pred_classes)
    test_auc = roc_auc_score(y_test, y_pred_probs) # Uses One-Hot y_test

    report_dict = classification_report(y_true_classes, y_pred_classes, target_names=['Benign', 'Illness'], output_dict=True)

    wandb.log({
        "test_accuracy": test_acc,
        "test_auc": test_auc,
        "test_recall_illness": report_dict['Illness']['recall'],
        "test_precision_illness": report_dict['Illness']['precision']
    })

    # B. Log Interactive ROC Curve (Crucial for Medical AI)
    # This creates a plot you can hover over in W&B
    wandb.log({"roc_curve": wandb.plot.roc_curve(
        y_true_classes,
        y_pred_probs,
        labels=["Benign", "Illness"]
    )})

    # C. Log Interactive Confusion Matrix
    wandb.log({"conf_mat": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_true_classes,
        preds=y_pred_classes,
        class_names=["Benign", "Illness"]
    )})

    print(classification_report(y_true_classes, y_pred_classes, target_names=['Benign', 'Illness']))
    wandb.finish()

Loading DermaMNIST...


In [8]:
def build_baseline_cnn():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 3)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(2, activation='softmax')
    ])
    return model

def build_vgg_style():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 3)),
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        # Head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(2, activation='softmax')
    ])
    return model

def build_resnet_style():
    # Helper: The Residual Block (The heart of ResNet)
    def res_block(x, filters, stride=1):
        shortcut = x

        # 1. Main Path (Conv -> BN -> ReLU -> Conv -> BN)
        x = layers.Conv2D(filters, (3, 3), strides=stride, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)

        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)

        # 2. Shortcut Path (Adjust dimensions if needed)
        # If we changed dimensions (stride > 1), we need to resize the shortcut too
        if stride > 1 or shortcut.shape[-1] != filters:
            shortcut = layers.Conv2D(filters, (1, 1), strides=stride, padding='same')(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)

        # 3. Add them together (The "Skip Connection")
        x = layers.Add()([x, shortcut])
        x = layers.Activation('relu')(x)
        return x

    # --- ARCHITECTURE START ---
    inputs = layers.Input(shape=(28, 28, 3))

    # Initial Conv
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Stack Residual Blocks (Gradually increasing depth)
    # 28x28 -> 28x28
    x = res_block(x, 32)
    x = res_block(x, 32)

    # 28x28 -> 14x14 (stride=2)
    x = res_block(x, 64, stride=2)
    x = res_block(x, 64)

    # 14x14 -> 7x7 (stride=2)
    x = res_block(x, 128, stride=2)
    x = res_block(x, 128)

    # Classification Head
    # Global Average Pooling is better than Flatten for ResNet
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(2, activation='softmax')(x)

    return models.Model(inputs, outputs)


In [6]:
# Experiment 1: Baseline
train_experiment(
    model_builder=build_baseline_cnn,
    exp_name="Exp1_Baseline",
    config={"batch_size": 128, "epochs": 10}
)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


55/55 ━━━━━━━━━━━━━━━━━━━━ 15s 177ms/step - accuracy: 0.4885 - auc: 0.5156 - loss: 0.6775 - val_accuracy: 0.5753 - val_auc: 0.6989 - val_loss: 0.5969
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 69ms/step - accuracy: 0.6409 - auc: 0.7623 - loss: 0.5305 - val_accuracy: 0.6670 - val_auc: 0.8064 - val_loss: 0.5151
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - accuracy: 0.6911 - auc: 0.8177 - loss: 0.5042 - val_accuracy: 0.7896 - val_auc: 0.9148 - val_loss: 0.3501
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 91ms/step - accuracy: 0.7379 - auc: 0.8599 - loss: 0.4454 - val_accuracy: 0.7458 - val_auc: 0.8755 - val_loss: 0.4316
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step - accuracy: 0.7498 - auc: 0.8624 - loss: 0.4402 - val_accuracy: 0.8046 - val_auc: 0.9164 - val_loss: 0.3640
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 69ms/step - accuracy: 0.7791 - auc: 0.8767 - loss: 0.4099 - val_accuracy: 0.7896 - val_auc: 0.8979 - val_loss: 0.3953
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 90ms/step - ac

epoch/accuracy,▁▄▅▆▇█████
epoch/auc,▁▅▆▇▇█████
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▅▄▃▂▁▂▁▁▁
epoch/val_accuracy,▁▃▆▅▆▆█▇▇▇
epoch/val_auc,▁▄▇▆▇▇████
epoch/val_loss,█▆▂▄▃▃▂▁▂▂
test_accuracy,▁
test_auc,▁
+2,...


In [7]:
# Experiment 2: VGG Style
train_experiment(
    model_builder=build_vgg_style,
    exp_name="Exp2_VGG_Deep",
    config={"batch_size": 128, "epochs": 30}
)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


55/55 ━━━━━━━━━━━━━━━━━━━━ 14s 172ms/step - accuracy: 0.6030 - auc: 0.6860 - loss: 0.6185 - val_accuracy: 0.7717 - val_auc: 0.8939 - val_loss: 0.3829
Epoch 2/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 72ms/step - accuracy: 0.7108 - auc: 0.8402 - loss: 0.4673 - val_accuracy: 0.7328 - val_auc: 0.8601 - val_loss: 0.4444
Epoch 3/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 92ms/step - accuracy: 0.6882 - auc: 0.7999 - loss: 0.4934 - val_accuracy: 0.6411 - val_auc: 0.7944 - val_loss: 0.5157
Epoch 4/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step - accuracy: 0.7215 - auc: 0.8347 - loss: 0.4894 - val_accuracy: 0.7228 - val_auc: 0.8685 - val_loss: 0.4237
Epoch 5/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 77ms/step - accuracy: 0.7222 - auc: 0.8373 - loss: 0.4490 - val_accuracy: 0.8016 - val_auc: 0.9113 - val_loss: 0.3598
Epoch 6/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 90ms/step - accuracy: 0.7809 - auc: 0.8848 - loss: 0.4020 - val_accuracy: 0.8754 - val_auc: 0.9435 - val_loss: 0.3017
Epoch 7/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 73ms/step - ac

epoch/accuracy,▁▄▃▄▅▆▆▆▇▇▇▇▇▇▇▇▆█▇███▇████▇█▇
epoch/auc,▁▄▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇███▇████▇██
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▅▄▄▄▄▃▃▃▃▂▂▃▂▃▂▂▂▁▂▂▂▂▂▂▂▁▂
epoch/val_accuracy,▅▄▁▃▅▇▄▅▆▆▇▅▇▅▇▇▇█▆█▆▆▆▇▆▇█▇█▆
epoch/val_auc,▅▄▁▄▆▇▄▅▅▆▇▄▆▅▇▇▇█▆█▆▆▆▇▆▇█▇█▇
epoch/val_loss,▅▆█▆▄▃▇▆▅▄▂▇▄▆▃▃▂▁▄▂▄▅▅▃▄▂▁▄▂▄
test_accuracy,▁
test_auc,▁
+2,...


In [9]:
train_experiment(
    model_builder=build_resnet_style,
    exp_name="Exp3_ResNet_Custom",
    config={"batch_size": 128, "epochs": 30, "architecture": "ResNet_Custom"}
)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


55/55 ━━━━━━━━━━━━━━━━━━━━ 36s 337ms/step - accuracy: 0.6415 - auc: 0.7199 - loss: 0.7245 - val_accuracy: 0.0987 - val_auc: 0.1060 - val_loss: 1.5805
Epoch 2/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 83ms/step - accuracy: 0.7522 - auc: 0.8563 - loss: 0.4492 - val_accuracy: 0.0987 - val_auc: 0.0946 - val_loss: 3.1085
Epoch 3/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 84ms/step - accuracy: 0.7810 - auc: 0.8798 - loss: 0.4083 - val_accuracy: 0.0987 - val_auc: 0.1066 - val_loss: 2.8973
Epoch 4/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 6s 104ms/step - accuracy: 0.7991 - auc: 0.8929 - loss: 0.4135 - val_accuracy: 0.0987 - val_auc: 0.1039 - val_loss: 2.6360
Epoch 5/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 85ms/step - accuracy: 0.8079 - auc: 0.9055 - loss: 0.3802 - val_accuracy: 0.0987 - val_auc: 0.1238 - val_loss: 2.5193
Epoch 6/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 97ms/step - accuracy: 0.8162 - auc: 0.9067 - loss: 0.3710 - val_accuracy: 0.1306 - val_auc: 0.1397 - val_loss: 2.4935
Epoch 7/30
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 89ms/step - a

epoch/accuracy,▁▃▄▅▆▅▆▆▆▇▆▆▇▆▆▇▇▇▇▇█▇▇████▇██
epoch/auc,▁▄▅▅▆▅▆▆▆▇▇▇▇▇▇▇▇▇▇███████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▅▅▅▄▄▃▃▃▃▃▂▃▂▃▂▂▂▂▁▂▂▂▂▁▁▁▁▁▁
epoch/val_accuracy,▁▁▁▁▁▁▂▃▄▇█▇█▇▆█▆▆▇▇▇█▆▇▆███▇█
epoch/val_auc,▁▁▁▁▁▁▂▂▄█████▆█▇▆▇▇▇█▆█▆█████
epoch/val_loss,▄█▇▇▇▇▆▄▃▁▁▁▁▁▂▁▂▂▂▂▂▁▂▂▂▁▁▁▁▁
test_accuracy,▁
test_auc,▁
+2,...
